[Homework 3]('https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/03-orchestration/hw-03.ipynb')

## Homework: AI Orchestration with Kestra

ATTENTION: At the end of the submission form, you will be required to include a link to your GitHub repository or other public code-hosting site. This repository should contain your code for solving the homework. If your solution includes code that is not in file format, please include these directly in the README file of your repository.

> In case you don't get one option exactly, select the closest one

## Prerequisites

Before starting this homework, ensure you have:

1. Completed the [Module 3 lessons](../../../03-orchestration/README.md) — the questions reference flows and concepts covered there
2. Kestra running locally with API keys configured (see the [Setup](../../../03-orchestration/lessons/03-setup.md) lesson) -- this includes the Gemini API key, which is also required for the AI Copilot
3. Imported all flows from the `03-orchestration/flows/` directory (covered in the Setup lesson)

## Assignment Overview

This homework explores AI workflows and agents in Kestra. You'll experiment with:
- Context engineering and why it matters
- RAG (Retrieval Augmented Generation) for grounded AI responses
- AI Agents for autonomous task execution
- Multi-agent systems for complex AI workflows

## Setup Instructions

Configure your API keys as Kestra secrets:

⚠️ Important: Never commit API keys to Git! Always use environment variables.

Kestra reads secrets from environment variables prefixed with `SECRET_` where the value is base64-encoded. Export your keys before starting Kestra:

```bash
export GEMINI_API_KEY="your-gemini-api-key-here"
export SECRET_GEMINI_API_KEY=$(echo -n "your-gemini-api-key-here" | base64)
export SECRET_TAVILY_API_KEY=$(echo -n "your-tavily-api-key-here" | base64)  # optional
```

Then start (or restart) Kestra:

```bash
cd 03-orchestration
docker compose up -d
```

In flows, secrets are referenced with `{{ secret('GEMINI_API_KEY') }}` — omit the `SECRET_` prefix when calling `secret()`.

Import the homework flows:

```bash
cd 03-orchestration

# Adjust username and password to match your Kestra setup
curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/1_chat_without_rag.yaml
curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/2_chat_with_rag.yaml
curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/4_simple_agent.yaml
curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/5_web_research_agent.yaml
curl -X POST -u 'admin@kestra.io:Admin1234!' http://localhost:8080/api/v1/flows/import -F fileUpload=@flows/6_multi_agent_research.yaml
```


## Question 1: Context Engineering

Try the following experiment:

1. Open ChatGPT in a private browser window: https://chatgpt.com
2. Enter this prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"
3. Then, use Kestra's AI Copilot with the same prompt

After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?

- AI Copilot uses a more powerful model
- `AI Copilot has access to current Kestra plugin documentation`
- AI Copilot uses more tokens
- AI Copilot has internet access

1. Open ChatGPT in a private browser window and enter the following prompt:

   **Prompt:**

   > Create a Kestra flow that loads NYC taxi data from CSV to BigQuery

2. ChatGPT should generate the following Kestra flow:

```yaml
id: load_nyc_taxi_csv_to_bigquery_gpt
namespace: zoomcamp

inputs:
  - id: csv_url
    type: STRING
    defaults: "https://raw.githubusercontent.com/DataTalksClub/nyc-tlc-data/master/trip%20data/yellow_tripdata_2019-01.csv"
  - id: gcp_project_id
    type: STRING
  - id: bq_dataset
    type: STRING
    defaults: nyc_taxi
  - id: bq_table
    type: STRING
    defaults: yellow_trips

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: "{{ inputs.csv_url }}"

  - id: create_table
    type: io.kestra.plugin.gcp.bigquery.CreateTable
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT_JSON') }}"
    projectId: "{{ inputs.gcp_project_id }}"
    dataset: "{{ inputs.bq_dataset }}"
    table: "{{ inputs.bq_table }}"
    tableDefinition:
      type: TABLE
      schema:
        fields:
          - { name: VendorID, type: INT64 }
          - { name: tpep_pickup_datetime, type: TIMESTAMP }
          - { name: tpep_dropoff_datetime, type: TIMESTAMP }
          - { name: passenger_count, type: INT64 }
          - { name: trip_distance, type: FLOAT64 }
          - { name: RatecodeID, type: INT64 }
          - { name: store_and_fwd_flag, type: STRING }
          - { name: PULocationID, type: INT64 }
          - { name: DOLocationID, type: INT64 }
          - { name: payment_type, type: INT64 }
          - { name: fare_amount, type: FLOAT64 }
          - { name: extra, type: FLOAT64 }
          - { name: mta_tax, type: FLOAT64 }
          - { name: tip_amount, type: FLOAT64 }
          - { name: tolls_amount, type: FLOAT64 }
          - { name: improvement_surcharge, type: FLOAT64 }
          - { name: total_amount, type: FLOAT64 }
          - { name: congestion_surcharge, type: FLOAT64 }

  - id: load_csv_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    serviceAccount: "{{ secret('GCP_SERVICE_ACCOUNT_JSON') }}"
    from: "{{ outputs.download_csv.uri }}"
    destinationTable: "{{ inputs.gcp_project_id }}.{{ inputs.bq_dataset }}.{{ inputs.bq_table }}"
    format: CSV
    writeDisposition: WRITE_TRUNCATE
    autodetect: false
    csvOptions:
      skipLeadingRows: 1
      fieldDelimiter: ","
      allowQuotedNewLines: true
```


3. Open **Kestra AI Copilot** and enter the same prompt.

   **Prompt:**

   > Create a Kestra flow that loads NYC taxi data from CSV to BigQuery

4. Kestra AI Copilot generates the following flow:

```yaml
id: load_nyc_taxi_csv_to_bigquery_copilot
namespace: zoomcamp

inputs:
  - id: csvFile
    type: FILE
    description: The NYC taxi data CSV file to load into BigQuery

tasks:
  - id: load_nyc_taxi_data_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ inputs.csvFile }}"
    destinationTable: "your_project_id.your_dataset.nyc_taxi_data" # Update with your BigQuery project, dataset, and table
    format: CSV
    csvOptions:
      skipLeadingRows: 1 # Assuming the CSV has a header row
    projectId: "your_project_id" # Update with your GCP Project ID
    createDisposition: CREATE_IF_NEEDED
    writeDisposition: WRITE_APPEND
```


### Експеримент

Порівняйте **ChatGPT** (у приватному вікні браузера) та **Kestra AI Copilot**, використовуючи однаковий запит:

> **"Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"**

### Чому AI Copilot генерує кращі Kestra-потоки?

**Відповідь:**
**Тому що AI Copilot має доступ до актуальної документації плагінів Kestra.**

### Пояснення

Це ключова ідея розділу **Context Engineering** у Модулі 3.

Звичайна велика мовна модель, така як ChatGPT, знає про Kestra лише те, що було доступне під час її навчання. Через це її знання можуть бути **застарілими або неповними**. У результаті модель може:

* вигадувати назви плагінів;
* використовувати застарілий синтаксис;
* генерувати потоки, які не працюють.

Натомість **Kestra AI Copilot** використовує **RAG (Retrieval-Augmented Generation)** або інші методи **Context Engineering**, щоб підставляти в контекст моделі **актуальну документацію плагінів Kestra**.

Завдяки цьому Copilot може:

* використовувати реальні назви плагінів;
* застосовувати актуальний синтаксис;
* генерувати коректні конфігурації плагінів;
* створювати потоки, які відповідають поточній версії Kestra.

**Висновок:** різницю визначає не "потужність" моделі чи кількість токенів, а **наявність достовірного та актуального контексту**, на основі якого модель формує відповідь.


In [ ]:
Q1: The primary reason Kestra's AI Copilot generates better Kestra flows is that it has access to current Kestra plugin documentation. This specialized knowledge allows it to produce more accurate and relevant flows specific to the Kestra platform.

## Question 2: RAG Comparison

Run both `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml` and compare their outputs. Both ask: "Which features were released in Kestra 1.1?"

What difference do you observe?

- Accurate and specific, matching the actual release notes
- `Vague, generic, or fabricated — the model guesses from training data`
- Empty — the model refuses to answer without context
- Identical to the RAG version

### Flows run: 1_chat_without_rag.yaml
❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting features that significantly enhanced its capabilities and user experience. Here are 5 major features with brief descriptions:
1. **Event-Driven Flows (Triggers for External Events):** This was a huge step forward for Kestra, enabling it to react to events happening outside of Kestra itself. Users could now define triggers that would start a flow based on external system events (e.g., a file appearing in S3, a message arriving in Kafka, or an HTTP request). This fundamentally transformed Kestra from primarily a scheduled orchestrator to a powerful event-driven automation platform.
2. **Global Kestra Variables:** Kestra 1.1 introduced the concept of global variables, allowing users to define variables once and make them accessible across all flows within a Kestra instance. This greatly improved maintainability and consistency, especially for common configurations like API keys, database connection strings, or shared resource identifiers. Instead of hardcoding or repeatedly defining these in multiple flows, they could be managed centrally.
3. **Task Outputs in UI (Execution Details):** This feature significantly improved the debugging and monitoring experience. For each task within an execution, users could now inspect the specific outputs generated by that task directly within the Kestra UI. This provided invaluable insight into what each task produced, making it much easier to understand the flow's progress, diagnose issues, and verify task results without needing to dig through logs or external systems.
4. **Flow-Level Labels:** Building upon the existing task-level labels, Kestra 1.1 extended this functionality to the flow level. Users could now assign labels to entire flows. This feature was crucial for organization, filtering, and potentially for more advanced use cases like grouping related flows, applying policies based on labels, or integrating with external monitoring/reporting tools.
5. **New Plugin for Microsoft Azure Services (Azure Blob Storage, Azure Kubernetes Service):** Kestra continued to expand its ecosystem of integrations with major cloud providers. Kestra 1.1 introduced a new official plugin providing native tasks for interacting with key Microsoft Azure services. This included tasks for managing files in Azure Blob Storage and interacting with Azure Kubernetes Service, opening up Kestra's automation capabilities to a wider range of Azure-centric environments and workflows.
🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?
👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.

### Flows run: 2_chat_with_rag.yaml
✅ RAG Response (with retrieved context):
Kestra 1.1 introduced several major features, including:
1. **New Filters**: The UI filters across Kestra were completely redesigned based on user feedback. The new design is cleaner, more intuitive, and more powerful, offering explicit filter options, single-click resets, saved filter combinations, and the ability to hide, show, or reorder table columns.
2. **No-Code Dashboard Editor**: Kestra 1.1 extended the No-Code Multi-Panel Editor to custom dashboards, allowing users to build and customize dashboards directly from the UI without writing YAML. This enables the creation of data sources, visualizations, and charts using form-based tabs.
3. **Human Task**: This new feature allows Enterprise Edition users to add manual approval steps to workflows. Executions reaching a human task pause until designated users or group members approve them, enabling human-in-the-loop workflows for critical operations.
4. **Multi-Agent AI Systems**: AI agents in Kestra can now use other AI agents as tools, facilitating sophisticated multi-agent orchestration workflows where a primary agent can delegate subtasks to specialized expert agents.
5. **Fix with AI**: When task runs fail, Kestra 1.1 can provide AI-powered suggestions to help diagnose and resolve issues. This feature analyzes failed task runs and offers intelligent recommendations for fixing problems, speeding up troubleshooting.
🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!

### Виконані потоки

* `1_chat_without_rag.yaml`
* `2_chat_with_rag.yaml`

### Запит

> **"Which features were released in Kestra 1.1? Please list at least 5 major features with brief descriptions."**

### Як найкраще описати відповідь без RAG?

**Відповідь:**
**Загальна, розмита або вигадана — модель робить припущення на основі навчальних даних.**

### Пояснення

Потік `1_chat_without_rag.yaml` навмисно містить у своєму результаті такі підказки:

> 🤔 **Чи помітили ви, що ця відповідь:**
>
> * ❌ містить неточності?
> * ❌ є надто загальною та розмитою?
> * ❌ перелічує функції, які були додані не саме у версії 1.1, а значно раніше?

Без механізму отримання контексту (retrieval) модель змушена покладатися лише на свої навчальні дані. Оскільки інформація про реліз Kestra 1.1 або не була присутня в навчальному корпусі, або відтворюється неточно, модель **галюцинує** (конфабулює) — генерує правдоподібні, але фактично некоректні списки функцій.

### Як `2_chat_with_rag.yaml` вирішує цю проблему

Потік `2_chat_with_rag.yaml`:

1. отримує актуальну статтю про реліз **Kestra 1.1** з GitHub;
2. індексує її у векторному сховищі **KestraKVStore**;
3. знаходить найбільш релевантні фрагменти документа;
4. передає їх як контекст моделі Gemini, завдяки чому вона формує **точну, перевірену та засновану на документації відповідь**.


In [ ]:
Q2: The difference you should observe is that the RAG version provides specific, accurate feature details grounded in the documentation. This is the primary advantage of using RAG in AI systems, as it reduces hallucination and improves factual accuracy by leveraging external knowledge.

## Question 3: Token Usage  — short summary

Run `4_simple_agent.yaml` twice:

1. First with `summary_length` = `short`
2. Second with `summary_length` = `long`

Check the token usage logged at the end of each execution. How does token usage differ between short and long summaries for the `multilingual_agent` task?

Run flows `4_simple_agent.yaml` with `summary_length` = `short`

📊 Token Usage Summary:

Summary Length Mode: short

Multilingual Agent:
- Input tokens: 282
- Output tokens: 90
- Total tokens: 372

English Brevity Agent:
- Input tokens: 105
- Output tokens: 43
- Total tokens: 148

💡 Tip: Monitor token usage to understand costs and optimize prompts!

Run flows `4_simple_agent.yaml` with `summary_length` = `medium`

📊 Token Usage Summary:

Summary Length Mode: medium

Multilingual Agent:
- Input tokens: 282
- Output tokens: 117
- Total tokens: 399

English Brevity Agent:
- Input tokens: 132
- Output tokens: 41
- Total tokens: 173

💡 Tip: Monitor token usage to understand costs and optimize prompts!

In [ ]:
Answer: For the multilingual_agent task, when comparing 'short' and 'long' summaries, you'll observe that the long summary typically uses 2-4x more output tokens than the short summary. This is because generating more detailed content requires the language model to produce a greater number of tokens.

### Q3: What is the approximate output token count for multilingual_agent when running with summary_length = short?
- 5-15 tokens
- `60-100 tokens`
- 200-400 tokens
- 500+ tokens

### Потік

`4_simple_agent.yaml` із параметром:

```text
summary_length = short
```

Потік складається з двох послідовних AI-агентів:

1. **`multilingual_agent`** — створює короткий підсумок вхідного тексту відповідно до системної інструкції:

   * Для `short`: **1–2 речення**.

2. **`english_brevity`** — скорочує цей підсумок до **рівно одного речення англійською мовою**.

Наприкінці задача **`log_token_usage`** виводить статистику використання токенів (вхідні, вихідні та загальні) для обох агентів.

### Вхідний текст

За замовчуванням використовується технічний опис Kestra обсягом приблизно **130 слів (~170 токенів)**.

### Приблизна кількість вихідних токенів для `multilingual_agent`

✅ **Відповідь:** **60–100 токенів**

### Пояснення

Підсумок із **1–2 речень** для технічного тексту обсягом близько **130 слів** зазвичай містить **30–80 токенів**, залежно від моделі.

Модель **Gemini 2.5 Flash** часто генерує трохи докладніші відповіді навіть за умови короткої інструкції, додаючи уточнювальні фрази. Хоча вимога обмежує відповідь до 1–2 речень, фактичний обсяг зазвичай становить **40–100 вихідних токенів**.

Тому найбільш відповідний діапазон — **60–100 токенів**.


In [ ]:
Q3: 60-100 tokens

## Question 4: Token Usage  — long summary

With summary_length = long, roughly how many times more output tokens does multilingual_agent use compared to the short summary?


- About the same (within 20%)
- `2-5x more`
- 10-20x more
- 50x more

Run flows `4_simple_agent.yaml` with `summary_length` = `long`

📊 Token Usage Summary:

Summary Length Mode: long

Multilingual Agent:
- Input tokens: 282
- Output tokens: 183
- Total tokens: 465

English Brevity Agent:
- Input tokens: 198
- Output tokens: 43
- Total tokens: 241

💡 Tip: Monitor token usage to understand costs and optimize prompts!

### Потік

`4_simple_agent.yaml` із параметром:

```text
summary_length = long
```

Системна інструкція для довгого підсумку:

> **"For 'long': 1-3 paragraphs"**

### Порівняння з коротким підсумком

Для `short`:

* Формат: **1–2 речення**
* Орієнтовне використання: **60–100 вихідних токенів**

Для `long`:

* Формат: **1–3 абзаци**
* Орієнтовне використання: **150–400 вихідних токенів**

### Питання

**У скільки разів більше вихідних токенів використовує `long` порівняно з `short`?**

✅ **Відповідь: 2–5× більше**

### Пояснення

Порівняння:

* Мінімальне співвідношення:

  * `150 / 100 = 1.5×`
  * це близько до нижньої межі очікуваного діапазону.

* Типове співвідношення:

  * `250 / 75 ≈ 3.3×`

* Максимальне співвідношення:

  * `400 / 60 ≈ 6.7×`

Основна реалістична оцінка для порівняння **1–3 абзаців** із **1–2 реченнями** — це приблизно **2–5 разів більше вихідних токенів**.


In [ ]:
Q4: 2-5x more

## Question 5: Modifying a flow

After changing english_brevity to ask for 3 sentences instead of 1, how does the output token count compare to the original 1-sentence version?


- About the same (within 20%)
- `2-4x more`
- 5-10x more
- 10x+ more

📊 Token Usage Summary:

Summary Length Mode: long

Multilingual Agent:
- Input tokens: 282
- Output tokens: 187
- Total tokens: 469

English Brevity Agent:
- Input tokens: 202
- Output tokens: 96
- Total tokens: 298

💡 Tip: Monitor token usage to understand costs and optimize prompts!

### Модифікація

У `4_simple_agent.yaml` змініть prompt для задачі **`english_brevity`**:

Було:

> Generate exactly 1 sentence English summary of the following:

Стає:

> Generate exactly 3 sentences English summary of the following:

Після цього запустіть потік із:

```text
summary_length = long
```

та порівняйте кількість вихідних токенів задачі **`english_brevity`** з оригінальною версією, яка генерувала 1 речення.

### Питання

**Як зміниться кількість вихідних токенів `english_brevity`?**

✅ **Відповідь: у 2–4 рази більше**

### Пояснення

Задача `english_brevity` отримує як вхід довший підсумок від `multilingual_agent`, але сама має фіксовану інструкцію щодо довжини результату.

Порівняння:

| Версія                 | Очікуваний результат                   | Вихідні токени  |
| ---------------------- | -------------------------------------- | --------------- |
| Оригінал: 1 речення    | Короткий англійський підсумок          | ~20–40 токенів  |
| Модифікація: 3 речення | Більш розгорнутий англійський підсумок | ~60–120 токенів |

Співвідношення:

* приблизно `60 / 40 = 1.5×` у мінімальному випадку;
* типово `90 / 30 ≈ 3×`.

Тому очікуваний результат — **приблизно у 2–4 рази більше вихідних токенів**.

Важливо: кількість **вхідних токенів** для `english_brevity` майже не змінюється, оскільки вона завжди обробляє результат `multilingual_agent`. Змінюється саме кількість **output tokens**, оскільки модель тепер генерує 3 речення замість 1.


In [ ]:
Q5: 2-4x more

## Question 6: Best Practices

Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?

- Always use AI agents for maximum flexibility and adaptation
- `Use traditional task-based workflows for predictability and auditability`
- Use only RAG without agents for better performance
- Use web search tools exclusively to ensure current data

### Сценарій

**Виробничі процеси, які потребують детермінованих і повторюваних результатів із суворими вимогами відповідності (наприклад, фінансова звітність або галузі з жорстким регулюванням).**

### Який підхід є найбільш відповідним?

✅ **Відповідь: використовувати традиційні task-based workflows для передбачуваності та можливості аудиту.**

### Пояснення

Урок **Best Practices** з Модуля 3 розділяє підходи таким чином:

| Підхід                              | Коли використовувати                                                                                                               |
| ----------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **Традиційні task-based workflows** | Детерміновані, контрольовані та придатні для аудиту процеси — кожен крок записується, версіонується та може бути відтворений.      |
| **AI agents**                       | Відкриті, дослідницькі задачі, де гнучкість важливіша за повну повторюваність результату.                                          |
| **RAG**                             | Забезпечує відповіді на основі фактичного контексту, але формат і генерація результату все одно можуть бути недетермінованими.     |
| **Web search tools**                | Підходять для отримання актуальних даних — можуть бути інструментом всередині агента, але не замінюють структуру робочого процесу. |

Для фінансової звітності або регульованих сфер **недетермінованість є ризиком**. AI-агент може:

* обирати різні послідовності дій під час різних запусків;
* генерувати різні формулювання результатів;
* потенційно створювати неточну інформацію.

Традиційні **Kestra task-based flows** забезпечують:

* однаковий шлях виконання при кожному запуску;
* повні журнали виконання для аудиту;
* контрольованість і відтворюваність процесів;
* простішу перевірку для внутрішніх команд та регуляторів.

**Висновок:** для критичних бізнес-процесів із вимогами відповідності слід обирати **детерміновані workflow-підходи**, а AI-агентів використовувати там, де потрібні гнучкість і адаптивність.


In [ ]:
Q6: For production workflows requiring deterministic, repeatable results with strict compliance requirements, the most appropriate approach is to use traditional task-based workflows. These workflows are designed for predictability and auditability, which are essential in highly regulated industries.